# Train a TBK SAE on Gemma 3 270M

This notebook trains a custom sparse autoencoder on `google/gemma-3-270m-it`, attached near the middle of the transformer blocks. It mixes roughly 2M tokens from a Pile-style dataset with the local `samples/questions_train.txt` data.

The SAE uses top-bottom-k activation (`tbk`), which keeps the largest-magnitude feature pre-activations and preserves their signs.

In [3]:
from pathlib import Path
import random
import sys
from datasets import load_dataset
import torch
print(torch.cuda.is_available())
print(torch.version.cuda)
# from datasets import load_dataset

REQUESTED_DEVICE = "cpu"  # Options: "auto", "cpu", "cuda", "cuda:0", "mps", ...


PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "trainable_sae.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from trainable_sae import (
    HookPointSpec,
    SAEConfig,
    SAEConnector,
    TrainableSAE,
    load_hooked_transformer,
    resolve_device,
)

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

device = resolve_device(REQUESTED_DEVICE)
dtype = torch.float32
device, dtype

TypeError: 'NoneType' object is not iterable

## Configuration

The hook layer is computed from the loaded model so it stays in the middle even if the model config changes. For Gemma 3 270M this should land around the center block.

In [ ]:
import time
MODEL_NAME = "google/gemma-3-270m-it"

PILE_DATASET = "HuggingFaceFW/fineweb"
PILE_DATASET_CONFIG = "sample-10BT"

QUESTIONS_PATH = PROJECT_ROOT / "samples/questions_train.txt"

PILE_TOKEN_BUDGET = 50_000_000
QUESTION_REPEATS = 3
BATCH_TOKENS = 8192
MODEL_FORWARD_TEXTS = 128
CONTEXT_SIZE = 64
EXPANSION_FACTOR = 32
TOP_K = 50
LR = 1e-4
MIN_LR = 5e-6
WARMUP_STEPS = 100
MAX_STEPS = None
LOG_NONZERO_FEATURES = True
SCHEDULER_TOTAL_STEPS = MAX_STEPS or max(1, PILE_TOKEN_BUDGET // BATCH_TOKENS)

SAVE_DIR = PROJECT_ROOT / f"custom_saes/gemma3_270m_mid_resid_post_shrink_fineweb50m_questions{int(time.time())}"


In [ ]:
# # MODEL_NAME = "pythia-70m"
# # Gemma3 -270m-it is a good choice for quick experiments due to its smaller size and faster training times compared to larger models. It still provides a reasonable capacity for learning from the dataset while allowing for quicker iterations during development and testing.
# import time


# MODEL_NAME = "google/gemma-3-270m-it"
# PILE_DATASET = "monology/pile-10k"
# PILE_DATASET = "monology/pile-uncopyrighted"

# QUESTIONS_PATH = PROJECT_ROOT / "samples/questions_train.txt"

# PILE_TOKEN_BUDGET = 10_000_000
# QUESTION_REPEATS = 3
# BATCH_TOKENS = 4096
# MODEL_FORWARD_TEXTS = 64
# CONTEXT_SIZE = 64
# EXPANSION_FACTOR = 32
# TOP_K = 50
# LR = 1e-4
# MIN_LR = 1e-5
# WARMUP_STEPS = 100
# MAX_STEPS = None  # Set to a small integer like 10 for a quick smoke test.
# SCHEDULER_TOTAL_STEPS = MAX_STEPS or max(1, PILE_TOKEN_BUDGET // BATCH_TOKENS)

# SAVE_DIR = PROJECT_ROOT / f"custom_saes/gemma3_270m_mid_resid_post_shrink_pile10m_questions{int(time.time())}"

## Load Gemma 3 270M

You may need to be logged into Hugging Face and have accepted the Gemma license before this cell can download the model.

In [ ]:
model = load_hooked_transformer(
    MODEL_NAME,
    device=device,
    dtype=dtype,
)

mid_layer = model.cfg.n_layers // 2
hook_point = HookPointSpec(layer=mid_layer, site="resid_post").name
d_in = model.cfg.d_model

print(f"model:      {MODEL_NAME}")
print(f"layers:     {model.cfg.n_layers}")
print(f"hook point: {hook_point}")
print(f"d_in:       {d_in}")

Loading weights: 100%|██████████| 236/236 [00:00<00:00, 8031.09it/s]


Loaded pretrained model google/gemma-3-270m-it into HookedTransformer
Moving model to device:  cuda
model:      google/gemma-3-270m-it
layers:     18
hook point: blocks.9.hook_resid_post
d_in:       640


## Build the SAE and connector

In [ ]:
# sae_cfg = SAEConfig(
#     d_in=d_in,
#     d_sae=d_in * EXPANSION_FACTOR,
#     activation="tbk",
#     k=TOP_K,
#     lr=LR,
#     l1_coefficient=0.0,  # TBK already enforces exactly k active features.
#     dtype="float32",    # Keep SAE training in fp32 even if the model is bf16.
#     device=device,
#     metadata={
#         "model_name": MODEL_NAME,
#         "hook_point": hook_point,
#         "pile_dataset": PILE_DATASET,
#         "pile_token_budget": PILE_TOKEN_BUDGET,
#         "questions_path": str(QUESTIONS_PATH),
#     },
# )

sae_cfg = SAEConfig(
    d_in=d_in,
    d_sae=d_in * EXPANSION_FACTOR,
    activation="shrink",
    shrink_threshold=0.5,
    lr=LR,
    l1_coefficient=1,  # Shrink already thresholds small feature activations to zero.
    dtype="float32",    # Keep SAE training in fp32 even if the model is bf16.
    device=device,
    metadata={
        "model_name": MODEL_NAME,
        "hook_point": hook_point,
        "pile_dataset": PILE_DATASET,
        "pile_token_budget": PILE_TOKEN_BUDGET,
        "questions_path": str(QUESTIONS_PATH),
    },
)

sae = TrainableSAE(sae_cfg, device=device)
connector = SAEConnector(model=model, sae=sae, hook_point=hook_point, device=device)
optimizer = torch.optim.AdamW(sae.parameters(), lr=sae_cfg.lr)

warmup_steps = min(WARMUP_STEPS, max(0, SCHEDULER_TOTAL_STEPS - 1))
cosine_steps = max(1, SCHEDULER_TOTAL_STEPS - warmup_steps)
warmup_scheduler = torch.optim.lr_scheduler.LinearLR(
    optimizer,
    start_factor=0.1,
    end_factor=1.0,
    total_iters=max(1, warmup_steps),
)
cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=cosine_steps,
    eta_min=MIN_LR,
)
scheduler = torch.optim.lr_scheduler.SequentialLR(
    optimizer,
    schedulers=[warmup_scheduler, cosine_scheduler],
    milestones=[warmup_steps],
)

print(
    f"SAE: {sae_cfg.d_in} -> {sae_cfg.d_sae}, activation={sae_cfg.activation}, "
    f"shrink_threshold={sae_cfg.shrink_threshold}"
)
print(
    f"scheduler: warmup={warmup_steps} steps, cosine={cosine_steps} steps, "
    f"lr={LR:g}->{MIN_LR:g}"
)

Moving model to device:  cuda
SAE: 640 -> 20480, activation=shrink, shrink_threshold=0.5
scheduler: warmup=100 steps, cosine=6003 steps, lr=0.0001->5e-06


## Prepare training text

`NeelNanda/pile-10k` is the same Pile-style source used elsewhere in this repo. The helper below takes approximately 2M model tokens from it, then mixes in the local generated question data.

In [ ]:
def read_questions(path: Path, repeats: int = 1):
    lines = [line.strip() for line in path.read_text().splitlines() if line.strip()]
    return lines * repeats


def count_activation_tokens_batch(texts, context_size: int = CONTEXT_SIZE):
    encoded = model.tokenizer(
        texts,
        add_special_tokens=True,
        truncation=True,
        max_length=context_size,
        return_length=True,
    )
    if "length" in encoded:
        return list(encoded["length"])
    return [len(input_ids) for input_ids in encoded["input_ids"]]


def count_activation_tokens(text: str, context_size: int = CONTEXT_SIZE) -> int:
    return count_activation_tokens_batch([text], context_size=context_size)[0]


def pile_texts_until_token_budget(dataset_name: str, token_budget: int, count_batch_size: int = 256):
    dataset = load_dataset(dataset_name, split="train", streaming=True)
    print('loaded dataaset')
    batch = []
    total_tokens = 0
    for row in dataset:
        text = row.get("text", "")
        if not text.strip():
            continue
        batch.append(text)
        if len(batch) < count_batch_size:
            continue

        counts = count_activation_tokens_batch(batch)
        for batch_text, count in zip(batch, counts):
            total_tokens += count
            yield batch_text, count
            if total_tokens >= token_budget:
                print(f"Collected about {total_tokens:,} truncated Pile activation tokens.")
                return
        batch = []

    if batch:
        counts = count_activation_tokens_batch(batch)
        for batch_text, count in zip(batch, counts):
            total_tokens += count
            yield batch_text, count
            if total_tokens >= token_budget:
                print(f"Collected about {total_tokens:,} truncated Pile activation tokens.")
                return
    print(f"Collected about {total_tokens:,} truncated Pile activation tokens.")


def repeat_to_token_budget(text_counts, token_budget: int):
    if not text_counts:
        return []

    repeated = []
    total_tokens = 0
    passes = 0
    while total_tokens < token_budget:
        passes += 1
        for text, count in text_counts:
            repeated.append(text)
            total_tokens += count
            if total_tokens >= token_budget:
                break
    print(f"Expanded to about {total_tokens:,} activation tokens over {passes} pass(es).")
    return repeated


question_texts = read_questions(QUESTIONS_PATH, repeats=QUESTION_REPEATS)
pile_text_counts = list(pile_texts_until_token_budget(PILE_DATASET, PILE_TOKEN_BUDGET))
pile_texts = [text for text, _ in pile_text_counts]
question_counts = count_activation_tokens_batch(question_texts)

training_texts = repeat_to_token_budget(pile_text_counts + list(zip(question_texts, question_counts)), PILE_TOKEN_BUDGET)
random.shuffle(training_texts)

print(f"Pile examples:     {len(pile_texts):,}")
print(f"Question examples: {len(question_texts):,}")
print(f"Total examples:    {len(training_texts):,}")

d:\andyh\Documents\Projects\mines\MATH498-Sparse-Autoencoder-Manipulation\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\andyh\.cache\huggingface\hub\datasets--HuggingFaceFW--fineweb. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


loaded dataaset
Collected about 50,000,047 truncated Pile activation tokens.
Expanded to about 50,000,047 activation tokens over 1 pass(es).
Pile examples:     781,386
Question examples: 3,000
Total examples:    781,386


## Activation batching

This harvests activations from the chosen middle block and flattens `[batch, seq, d_model]` into token rows for SAE training.

In [ ]:
def batched(iterable, batch_size):
    batch = []
    for item in iterable:
        batch.append(item)
        if len(batch) == batch_size:
            yield batch
            batch = []
    if batch:
        yield batch


def collect_activations_for_text_batch(text_batch, context_size=CONTEXT_SIZE):
    encoded = model.tokenizer(
        text_batch,
        add_special_tokens=True,
        padding=True,
        truncation=True,
        max_length=context_size,
        return_tensors="pt",
    )
    tokens = encoded["input_ids"].to(device)
    attention_mask = encoded["attention_mask"].to(device)

    with torch.no_grad():
        _, cache = model.run_with_cache(
            tokens,
            names_filter=hook_point,
            attention_mask=attention_mask,
        )

    acts = cache[hook_point].detach().to(dtype=torch.float32)
    return acts[attention_mask.bool()].reshape(-1, d_in)


def activation_batches(
    texts,
    batch_tokens=BATCH_TOKENS,
    context_size=CONTEXT_SIZE,
    model_forward_texts=MODEL_FORWARD_TEXTS,
):
    buffer = []
    buffered = 0

    for text_batch in batched(texts, model_forward_texts):
        acts = collect_activations_for_text_batch(text_batch, context_size=context_size)
        buffer.append(acts)
        buffered += acts.shape[0]

        if buffered >= batch_tokens:
            joined = torch.cat(buffer, dim=0)
            yield joined[:batch_tokens]
            remainder = joined[batch_tokens:]
            buffer = [remainder] if remainder.numel() else []
            buffered = remainder.shape[0] if remainder.numel() else 0

    if buffer:
        yield torch.cat(buffer, dim=0)

## Train

In [ ]:
metrics = []
for step, batch in enumerate(activation_batches(training_texts), start=1):
    batch = batch.to(device=device, dtype=torch.float32)
    step_metrics = sae.training_step(
        batch,
        optimizer,
        record_nonzero_features=LOG_NONZERO_FEATURES,
    )
    scheduler.step()
    step_metrics["lr"] = scheduler.get_last_lr()[0]
    metrics.append(step_metrics)

    if step == 1 or step % 10 == 0:
        nonzero_text = ""
        if LOG_NONZERO_FEATURES:
            nonzero_text = f" | avg_nonzero={step_metrics['avg_nonzero_features']:.2f}"
        print(
            f"step {step:04d} | "
            f"loss={step_metrics['loss']:.5f} | "
            f"mse={step_metrics['mse']:.5f} | "
            f"l1={step_metrics['l1']:.5f}"
            f"{nonzero_text} | "
            f"lr={step_metrics['lr']:.2e}"
        )

    if MAX_STEPS is not None and step >= MAX_STEPS:
        break

print(f"Finished {len(metrics)} optimization steps.")

step 0001 | loss=2821945.75000 | mse=2821803.25000 | l1=142.44145 | lr=1.09e-05
step 0010 | loss=1462968.37500 | mse=1462826.37500 | l1=141.95280 | lr=1.90e-05
step 0020 | loss=395201.12500 | mse=395058.87500 | l1=142.24252 | lr=2.80e-05
step 0030 | loss=114122.64062 | mse=113979.83594 | l1=142.80115 | lr=3.70e-05
step 0040 | loss=29414.52734 | mse=29271.72461 | l1=142.80305 | lr=4.60e-05
step 0050 | loss=18316.70117 | mse=18174.43555 | l1=142.26607 | lr=5.50e-05
step 0060 | loss=11272.54590 | mse=11130.43945 | l1=142.10692 | lr=6.40e-05
step 0070 | loss=7384.02930 | mse=7243.41260 | l1=140.61662 | lr=7.30e-05
step 0080 | loss=4881.28955 | mse=4739.23682 | l1=142.05289 | lr=8.20e-05
step 0090 | loss=3273.13672 | mse=3131.58838 | l1=141.54831 | lr=9.10e-05
step 0100 | loss=3060.45190 | mse=2919.55151 | l1=140.90050 | lr=1.00e-04
step 0110 | loss=2117.11011 | mse=1975.29102 | l1=141.81912 | lr=1.00e-04
step 0120 | loss=1594.75671 | mse=1452.13623 | l1=142.62044 | lr=1.00e-04
step 0130 | 

## Load a saved SAE

In [ ]:
# Set this to a saved SAE directory, or leave it as None to keep the current in-memory SAE.
# Examples:
LOAD_SAE_PATH = PROJECT_ROOT / "gemma3_270m_four_saes_1777185250/shrink"
# LOAD_SAE_PATH = SAVE_DIR

# LOAD_SAE_PATH = None

if LOAD_SAE_PATH is not None:
    LOAD_SAE_PATH = Path(LOAD_SAE_PATH)
    sae = TrainableSAE.load(LOAD_SAE_PATH, device=device)
    sae_cfg = sae.cfg
    hook_point = sae_cfg.metadata.get("hook_point", hook_point)
    connector = SAEConnector(model=model, sae=sae, hook_point=hook_point, device=device)
    optimizer = torch.optim.AdamW(sae.parameters(), lr=sae_cfg.lr)
    print(f"Loaded SAE from {LOAD_SAE_PATH}")
    print(
        f"SAE: {sae_cfg.d_in} -> {sae_cfg.d_sae}, activation={sae_cfg.activation}, "
        f"hook_point={hook_point}"
    )
else:
    print("Using current in-memory SAE.")

## Save and smoke-test insertion

In [ ]:
import time

SAVE_DIR.mkdir(parents=True, exist_ok=True) 
sae.save(SAVE_DIR)
print(f"Saved SAE to {SAVE_DIR}")

prompt = "Sarah put the book in the kitchen. Answer in one word: where is the book?"
tokens = model.to_tokens(prompt)

with torch.no_grad():
    logits = connector.run_with_sae(tokens, mode="reconstruct")

print(logits.shape)
print("latest features:", connector.latest_features.shape if connector.latest_features is not None else None)

Saved SAE to d:\andyh\Documents\Projects\mines\MATH498-Sparse-Autoencoder-Manipulation\custom_saes\gemma3_270m_mid_resid_post_shrink_fineweb50m_questions1777167563
torch.Size([1, 19, 262144])
latest features: torch.Size([1, 19, 20480])


In [ ]:
EVAL_PROMPT = "tell me true or false: a square has five sides"
MAX_NEW_TOKENS = 50
TEMPERATURE = 0.0

generate_kwargs = dict(
    max_new_tokens=MAX_NEW_TOKENS,
    temperature=TEMPERATURE,
    verbose=False,
)

def clean_generation(text: str) -> str:
    return text.split("<end_of_turn>")[0].strip()

tokens = model.to_tokens(EVAL_PROMPT).to(device)
prompt_len = tokens.shape[1]

model.reset_hooks()
with torch.no_grad():
    normal_tokens = model.generate(tokens, **generate_kwargs)
normal_output = clean_generation(model.to_string(normal_tokens[0, prompt_len:]))

with torch.no_grad():
    sae_output = clean_generation(
        connector.generate_with_sae(EVAL_PROMPT, mode="reconstruct", **generate_kwargs)
    )

print("Prompt:")
print(EVAL_PROMPT)
print("\nNormal model:")
print(normal_output)
print("\nWith SAE inserted:")
print(sae_output)
print("\nLatest SAE feature tensor:", connector.latest_features.shape if connector.latest_features is not None else None)


Prompt:
tell me true or false: a square has five sides

Normal model:
.
A square has 4 sides.
A square has 4 sides.
A square has 4 sides.
A square has 4 sides.

I think the answer is:
**True**

Final Answer: **True

With SAE inserted:
.

**False**

A square has four sides.
**True**

A square has 4 sides.
**False**

A square has 4 sides.
**True**

A square has 4 sides.
**

Latest SAE feature tensor: torch.Size([1, 1, 20480])
